# Day 21：Week 3 综合测评与集成迷你项目（多轮记忆 + 工具调用 + 流式输出）

> **🎙️ 讲师旁白**：欢迎来到星辰数科《智枢企业级智能体平台》第二阶段第 21 天！今天我们将第三周学到的核心能力——**多轮上下文记忆（Multi-turn Memory）**、**Pure-API Function Calling（`get_weather` / `calculate_math`）** 与 **SSE 生成器流式输出（`stream_reply`）**——融合为统一的企业级 `IntegratedAgent` 引擎（`v0.2.0-day21`）。

## 1. 环境初始化与导入 `project_iterations/day_21` 模块
在无 `OPENAI_API_KEY` 的离线环境下，`IntegratedAgent` 会自动启用 `httpx.MockTransport` 模拟通道，完整演示工具决策与 SSE 流式吐字。

In [1]:
import os
import sys
from pathlib import Path

repo_root = Path.cwd()
while not (repo_root / "project_iterations").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
day21_dir = repo_root / "project_iterations" / "day_21"
if str(day21_dir) not in sys.path:
    sys.path.insert(0, str(day21_dir))

os.environ.pop("OPENAI_API_KEY", None)

import httpx
from zhishu_agent import IntegratedAgent, safe_calculate_math

agent = IntegratedAgent()
print("Mock 模式启用:", agent.is_mock)
print("已注册工具:", agent.registry.list_tools())

Mock 模式启用: True
已注册工具: ['get_weather', 'calculate_math']


## 2. 安全基线回顾：AST 白名单数学求值器
验证 `safe_calculate_math` 正确执行复杂算术运算，同时拦截恶意代码注入。

In [2]:
print("计算 (120 + 80) * 3 / 4 =", safe_calculate_math("(120 + 80) * 3 / 4"))

try:
    safe_calculate_math("__import__('os').system('echo pwned')")
except ValueError as exc:
    print("成功拦截恶意注入:", exc)

计算 (120 + 80) * 3 / 4 = 150
成功拦截恶意注入: 检测到不安全的表达式语法节点: Call


## 3. 多轮记忆 + 并行工具调用 + 生成器流式输出实战
通过 `agent.stream_reply()` 依次完成：身份记忆 -> 并行调用 `get_weather` 与 `calculate_math` -> 跨轮次记忆召回。

In [3]:
queries = [
    "你好，我叫陈默。",
    "帮我查一下北京天气，并计算 (120 + 80) * 3 / 4 等于多少？",
    "我叫什么名字？",
    "刚才我查了哪个城市的天气？",
]

for i, q in enumerate(queries, start=1):
    print(f"\n[第 {i} 轮] 用户: {q}")
    chunks = list(agent.stream_reply(q))
    print(f"[流式分片数={len(chunks)}] 助手: {''.join(chunks)}")
    if agent.last_run and agent.last_run.executed_tools:
        for tc in agent.last_run.executed_tools:
            print(f"  -> 执行工具 {tc['name']}({tc['arguments']}) => {tc['output']}")

print("\n消息角色序列:", [m["role"] for m in agent.get_history()])
print(f"总轮数: {agent.turn_count} | 累计 Tokens: {agent.total_tokens}")


[第 1 轮] 用户: 你好，我叫陈默。
[流式分片数=4] 助手: 您好，陈默！我已经记住您的名字，请问有什么可以帮您？

[第 2 轮] 用户: 帮我查一下北京天气，并计算 (120 + 80) * 3 / 4 等于多少？
[流式分片数=9] 助手: 已为您调用工具完成查询：北京当前天气晴朗微风，气温 18°C，湿度 35%；算式 (120 + 80) * 3 / 4 的计算结果为 150。
  -> 执行工具 get_weather({"city": "北京", "unit": "celsius"}) => {"city": "北京", "temperature": 18, "unit": "°C", "condition": "晴朗微风", "humidity": "35%"}
  -> 执行工具 calculate_math({"expression": "(120 + 80) * 3 / 4"}) => {"expression": "(120 + 80) * 3 / 4", "result": 150}

[第 3 轮] 用户: 我叫什么名字？
[流式分片数=2] 助手: 您刚才告诉我，您的名字是陈默。

[第 4 轮] 用户: 刚才我查了哪个城市的天气？
[流式分片数=2] 助手: 您上一轮查询天气的城市是北京。

消息角色序列: ['system', 'user', 'assistant', 'user', 'assistant', 'tool', 'tool', 'assistant', 'user', 'assistant', 'user', 'assistant']
总轮数: 4 | 累计 Tokens: 490


## 4. 事务性异常回滚验证
当服务端返回 `HTTP 500` 时，`IntegratedAgent` 应自动回滚本轮未提交的消息，确保已有对话历史不被污染。

In [4]:
def fail_handler(request: httpx.Request) -> httpx.Response:
    return httpx.Response(status_code=500, json={"error": "Server Error"})

err_agent = IntegratedAgent(transport=httpx.MockTransport(fail_handler))
before_len = len(err_agent.get_history())
try:
    list(err_agent.stream_reply("触发异常请求"))
except httpx.HTTPStatusError as exc:
    print("捕获预期 HTTP 异常:", exc.response.status_code)

after_len = len(err_agent.get_history())
print(f"回滚验证: 异常前消息数={before_len}, 异常后消息数={after_len}")

捕获预期 HTTP 异常: 500
回滚验证: 异常前消息数=1, 异常后消息数=1
